說明

把使用者輸入的日常瑣事，包裝成戲劇化的電影預告片旁白。流程：

1. Writer 寫第一版旁白
2. Reviewer 從戲劇張力、懸念度、澎湃感、轉折感四個面向打分並給建議
3. Writer 根據建議改寫第二版

這是 Reflection 模式：寫稿、自審、修稿。


## 套件說明

本作業未使用範例外套件，主要依賴兩個：

- `aisuite[all]`：LLM 統一介面，一套 API 可串接 Groq、OpenAI、Mistral 等。方便 Writer 和 Reviewer 接不同 provider。
- `gradio`：Web UI 框架。`gr.Blocks` 自訂版面、`gr.Examples` 提供範例按鈕、`gr.themes.Soft()` 套用主題。

預設用 Groq。


In [ ]:
import os
from google.colab import userdata


In [ ]:

#【使用 Groq】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
provider = "groq"
model = "openai/gpt-oss-120b"


In [ ]:
!pip install aisuite[all]


In [ ]:
import aisuite as ai


In [ ]:
provider_writer = "groq"
model_writer = "openai/gpt-oss-120b"

provider_reviewer = "groq"
model_reviewer = "openai/gpt-oss-120b"

# 也可以讓兩者用不同模型, 例如 reviewer 換成 openai gpt-4o
# provider_reviewer = "openai"
# model_reviewer = "gpt-4o"


In [ ]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider="groq",
          model="openai/gpt-oss-120b"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content


## 設定兩個角色

Reflection 模式的核心是兩個 system prompt：

- `system_writer` 定義 Writer 的寫作風格（短句、停頓、史詩感、反轉結尾）
- `system_reviewer` 定義 Reviewer 用四面向評分制給回饋

用評分制取代純文字建議，能逼 Reviewer 給出具體可量化的意見，回饋更結構化。


In [ ]:
system_writer = """你是好萊塢頂級電影預告片旁白創作大師，擅長把一件平凡瑣事或日常小事，用極度戲劇化、懸念感、史詩感、澎湃的口吻包裝成電影預告片旁白。

請遵守以下風格規則：
- 第三人稱、低沉嗓音的口吻
- 句子要短、節奏要快、停頓要多 (請用「……」表現停頓)
- 多用對比、轉折、命運、宿命、選擇、抉擇等史詩感詞彙
- 適時加入畫面感的描述 (例如：鏡頭緩緩拉近、雨水打在臉上、世界突然安靜)
- 結尾一定要有一個吊人胃口的反轉句, 讓人想知道接下來會發生什麼
- 請用台灣習慣的中文回覆"""

system_reviewer = """你是資深好萊塢電影預告片導演，看過上萬部預告片，眼光毒辣、講話直接。

請從以下四個面向, 評論這段預告片旁白, 並給出具體可執行的修改建議:

1. 🎭 戲劇張力 (情緒是否到位? 是否夠強烈?)
2. ⏳ 懸念度 (觀眾看完是否會想知道接下來會發生什麼?)
3. 🔥 澎湃感 (節奏感、史詩感是否夠?)
4. 🌪️ 轉折感 (是否有意料之外的反轉?)

請依照以下格式輸出評論:

【🎭 戲劇張力】X/10 分
(具體評論與建議)

【⏳ 懸念度】X/10 分
(具體評論與建議)

【🔥 澎湃感】X/10 分
(具體評論與建議)

【🌪️ 轉折感】X/10 分
(具體評論與建議)

【🎯 最需要改進的點】
(用一句話總結最關鍵的修改方向)

請用台灣習慣的中文回覆。"""


## Reflection 核心流程

`trailer_reflect()` 把三步驟串起來，回傳三個結果交給 Gradio 並排顯示。


In [ ]:
def trailer_reflect(prompt):
    # Step 1: Writer 寫第一版預告片旁白
    first_version = reply(system_writer,
                          f"請把以下這件事改寫成電影預告片旁白:\n{prompt}",
                          provider=provider_writer,
                          model=model_writer
                          )

    # Step 2: Reviewer (導演) 給四面向評分與建議
    suggestion = reply(system_reviewer, first_version,
                       provider=provider_reviewer,
                       model=model_reviewer
                       )

    # Step 3: Writer 根據導演建議, 改寫第二版
    second_prompt = (
        f"這是我剛剛寫的預告片旁白:\n{first_version}\n\n"
        f"這是導演大大的修改建議:\n{suggestion}\n\n"
        f"請根據這些建議, 幫我改寫一個更戲劇化、更澎湃、更有反轉的第二版。"
        f"請用台灣習慣的中文, 並且只要輸出改好的旁白文字就可以了, 不要其他說明。"
    )
    second_version = reply(system_writer, second_prompt,
                           provider=provider_writer,
                           model=model_writer
                           )

    return first_version, suggestion, second_version


## Gradio 介面


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr


In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 電影預告片旁白反思器 Movie Trailer Voice Reflection")
    gr.Markdown("> 輸入一件你今天發生的瑣事或無聊小事, AI 會用 Reflection 模式把它包裝成史詩級電影預告片旁白!")

    user_input = gr.Textbox(
        label="📝 今天發生了什麼?",
        placeholder="例：我今天忘了帶傘, 淋成落湯雞...",
        lines=2
    )
    btn = gr.Button("🎥 生成預告片旁白!", variant="primary", size="lg")

    with gr.Row():
        out1 = gr.Textbox(label="🎬 第一版預告片旁白 (Writer)", lines=10)
        out2 = gr.Textbox(label="🎯 導演評分與建議 (Reviewer)", lines=10)
        out3 = gr.Textbox(label="🏆 第二版終極預告片 (Writer 改寫)", lines=10)

    btn.click(trailer_reflect, inputs=[user_input], outputs=[out1, out2, out3])

    gr.Examples(
        examples=[
            "我今天忘了帶傘, 淋成落湯雞",
            "我剛剛在家門口踩到貓便便",
            "上班開會的時候打瞌睡被老闆抓包",
            "便利商店的微波便當熱完是冷的",
            "我喜歡的人已讀不回我三天了",
            "出門才發現襪子穿成不同顏色",
            "外送平台的麻辣鍋送來變冷了"
        ],
        inputs=user_input,
        label="💡 試試這些範例"
    )


In [ ]:
demo.launch(share=True, debug=True)
